# Mixed OpenRouter + Local/HF Collection

This notebook shows one way to collect EigenBench evaluations from a mixed model population (OpenRouter and local Hugging Face models), then optionally merge evaluation files.

## Notes

- Keep model nicknames stable across all evaluation files so merge/remap is reliable.
- Local models can be slow; start with small `dataset.count` and `group_size`.
- Install optional deps first if needed: `pip install transformers accelerate`.

In [ ]:
from __future__ import annotations

import importlib
from pathlib import Path

from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline

from pipeline.config import load_dataset_scenarios_from_spec, get_criteria_from_spec
from pipeline.eval.collect import collect_core_evaluations
import pipeline.eval.criteria_collectors as criteria_collectors
from pipeline.io import append_records, load_records
from pipeline.providers import get_openrouter_response

load_dotenv()

In [ ]:
# --- Configuration ---
SPEC_MODULE = "runs.example"
OUTPUT_PATH = "runs/example/out/evaluations_local_openrouter.jsonl"

# Add local models using the prefix `hf_local:`.
# Keep nicknames aligned with your run spec when you plan to train from merged data.
LOCAL_MODELS = {
    # "Llama 3.1 8B Local": "hf_local:meta-llama/Meta-Llama-3.1-8B-Instruct",
}

spec = importlib.import_module(SPEC_MODULE).RUN_SPEC
models = dict(spec["models"])  # OpenRouter models from run spec
models.update(LOCAL_MODELS)

print(f"Using {len(models)} models")
for nick, model_id in models.items():
    print(f"  - {nick}: {model_id}")

In [ ]:
_LOCAL_GENERATORS = {}


def _get_local_generator(model_id: str):
    if model_id in _LOCAL_GENERATORS:
        return _LOCAL_GENERATORS[model_id]

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    generator = hf_pipeline("text-generation", model=model, tokenizer=tokenizer)
    _LOCAL_GENERATORS[model_id] = (tokenizer, generator)
    return _LOCAL_GENERATORS[model_id]


def _render_prompt(messages, tokenizer):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        parts = []
        for m in messages:
            role = m.get("role", "user").upper()
            content = m.get("content", "")
            parts.append(f"{role}: {content}")
        parts.append("ASSISTANT:")
        return "\n\n".join(parts)


def mixed_get_model_response(model_name: str, messages, max_tokens: int, return_full_response: bool = False, log_probs: bool = False):
    if model_name.startswith("openrouter:"):
        upstream = model_name.split(":", 1)[1]
        return get_openrouter_response(
            messages=messages,
            model=upstream,
            max_tokens=max_tokens,
            return_full_response=return_full_response,
        )

    if model_name.startswith("hf_local:"):
        model_id = model_name.split(":", 1)[1]
        tokenizer, generator = _get_local_generator(model_id)
        prompt = _render_prompt(messages, tokenizer)
        out = generator(
            prompt,
            max_new_tokens=max_tokens,
            do_sample=False,
            return_full_text=False,
        )
        if not out:
            return ""
        return out[0].get("generated_text", "").strip()

    # Backward-compat: plain model IDs are treated as OpenRouter IDs.
    return get_openrouter_response(
        messages=messages,
        model=model_name,
        max_tokens=max_tokens,
        return_full_response=return_full_response,
    )


# Monkeypatch collection call path for this notebook session.
criteria_collectors.get_model_response = mixed_get_model_response

In [ ]:
ds = spec["dataset"]
cfg = spec["collection"]
constitution = spec["constitution"]

criteria = get_criteria_from_spec(constitution)
all_scenarios = load_dataset_scenarios_from_spec(ds)
start = int(ds.get("start", 0))
count = int(ds.get("count", len(all_scenarios) - start))
selected = all_scenarios[start : start + count]

existing = load_records(OUTPUT_PATH)

for offset, scenario in enumerate(selected):
    scenario_index = start + offset
    new_evals = collect_core_evaluations(
        criteria=criteria,
        scenario=scenario,
        scenario_index=scenario_index,
        models=models,
        evaluations=existing,
        sampler_mode=cfg.get("sampler_mode", "random_judge_group"),
        allow_ties=bool(cfg.get("allow_ties", True)),
        group_size=int(cfg.get("group_size", 4)),
        groups=int(cfg.get("groups", 1)),
        alpha=float(cfg.get("alpha", 2.0)),
    )
    append_records(OUTPUT_PATH, new_evals)
    existing.extend(new_evals)
    print(f"scenario_index={scenario_index}: wrote {len(new_evals)} rows")

print(f"Done. Output: {Path(OUTPUT_PATH).resolve()}")

## Merge with another evaluation file

After you have multiple evaluation files, merge and remap them with:

```bash
python scripts/run_merge_evaluations.py runs.example runs/example/out/evaluations_merged.jsonl runs/example/out/evaluations_or.jsonl runs/example/out/evaluations_local_openrouter.jsonl
```

Then point `collection.evaluations_path` to the merged file and run training as usual.